Synthetic NPCI/UPI FRM (Fraud Risk Management) log data generator.

Generates random data following the structure of the sample log lines:
  - reqPay request message (Payer/Payee details, device tags, accounts)
  - Acknowledgement for ReqRiskScore
  - FRM response message with RiskScores
  - reqPay request message (response="N" leg, orgRespCode)
  - Acknowledgement for ReqFrmAdvice

All values (IDs, hashes, IFSCs, amounts, timestamps, IPs, etc.) are
randomly generated / fake and are NOT derived from any real transaction.

In [1]:
import os
import json
import random
import string
import hashlib
import argparse
from datetime import datetime, timedelta, timezone
 
IST = timezone(timedelta(hours=5, minutes=30))
 

# ============================================================
# UPI / eFRM SYNTHETIC MASTER DATA
# ============================================================

# UPI apps and payment applications
APPS = [
    # Existing applications
    "com.phonepe.app",
    "net.one97.paytm",
    "in.org.npci.upiapp",
    "com.google.android.apps.nbu.paisa.user",
    "in.amazon.mShop.android.shopping",

    # Additional payment / banking applications
    "com.mobikwik_new",
    "com.freecharge",
    "com.myairtelapp",
    "com.bharatpe.app",
    "com.olacabs.customer",
    "com.whatsapp",
    "com.truecaller",
    "com.sbi.lotusintouch",
    "com.sbi.SBIFreedomPlus",
    "com.axis.mobile",
    "com.axis.mobilebanking",
    "com.snapwork.hdfc",
    "com.csam.icici.bank.imobile",
    "com.kotak.mobilebanking",
    "com.infrasofttech.indianbank",
    "com.fss.pnbpsp",
    "com.canarabank.mobility",
    "com.unionbank.ecommerce.mobile.android",
    "com.idbi.mpassbook",
    "com.bankofbaroda.mconnect",
    "com.federalbank.mobile",
    "com.yesbank.mobile",
    "com.indusind.mobile",
    "com.bandhan.mobilebanking",
    "com.rblbank.mobank",
    "com.au.bank",
    "com.jio.myjio",
    "com.flipkart.android",
    "com.myntra.android",
    "com.swiggy.android",
    "com.application.zomato",
]

# Operating systems
OS_LIST = [
    "Android",
    "iOS",
]

# Optional OS-version values for synthetic device profiles
OS_VERSIONS = {
    "Android": [
        "10", "11", "12", "12L", "13", "14", "15", "16"
    ],
    "iOS": [
        "16", "16.7", "17", "17.7", "18", "18.5", "26"
    ],
}

# Synthetic bank / issuer identifiers
# Format: (bank_code, IFSC_like_code)
BANK_IFSCS = [
    # Existing values
    ("KKBK", "KKBK0007531"),
    ("HDFC", "HDFC0001234"),
    ("ICIC", "ICIC0000456"),
    ("SBIN", "SBIN0011700"),
    ("JIOP", "JIOP0000001"),
    ("UTIB", "UTIB0000512"),

    # Additional synthetic entries
    ("BARB", "BARB0000001"),
    ("PUNB", "PUNB0000001"),
    ("CNRB", "CNRB0000001"),
    ("UBIN", "UBIN0000001"),
    ("IOBA", "IOBA0000001"),
    ("IDIB", "IDIB0000001"),
    ("IBKL", "IBKL0000001"),
    ("YESB", "YESB0000001"),
    ("INDB", "INDB0000001"),
    ("FDRL", "FDRL0000001"),
    ("RATN", "RATN0000001"),
    ("BDBL", "BDBL0000001"),
    ("ESFB", "ESFB0000001"),
    ("AUBL", "AUBL0000001"),
    ("KVBL", "KVBL0000001"),
    ("TMBL", "TMBL0000001"),
    ("SIBL", "SIBL0000001"),
    ("DCBL", "DCBL0000001"),
    ("JAKA", "JAKA0000001"),
    ("MAHB", "MAHB0000001"),
    ("PSIB", "PSIB0000001"),
    ("BKID", "BKID0000001"),
    ("UCBA", "UCBA0000001"),
    ("CBIN", "CBIN0000001"),
    ("ORBC", "ORBC0000001"),
    ("CORP", "CORP0000001"),
    ("ANDB", "ANDB0000001"),
    ("ALLA", "ALLA0000001"),
    ("SYNB", "SYNB0000001"),
    ("VIJB", "VIJB0000001"),
    ("LAVB", "LAVB0000001"),
    ("NSPB", "NSPB0000001"),
]

# UPI ID handles / PSP suffixes
HANDLES = [
    # Existing values
    "@ybl",
    "@axl",
    "@okhdfcbank",
    "@oksbi",
    "@okicici",
    "@paytm",
    "@ibl",

    # Additional commonly seen or illustrative suffixes
    "@upi",
    "@apl",
    "@jio",
    "@okaxis",
    "@okbob",
    "@okunionbank",
    "@okcanara",
    "@okpnb",
    "@okyes",
    "@okindus",
    "@okfederal",
    "@okkotak",
    "@okicici",
    "@okhdfcbank",
    "@okaxis",
    "@okbizaxis",
    "@oksbi",
    "@okbizsbi",
    "@okbizicici",
    "@okbizhdfcbank",
    "@freecharge",
    "@mobikwik",
    "@slice",
    "@ibl",
    "@idfcbank",
    "@indus",
    "@fbl",
    "@aubank",
    "@dbs",
    "@cnrb",
    "@barodampay",
    "@axisbank",
]

# Transaction purpose codes
PURPOSE_CODES = [
    "00", "01", "02",
    "03", "04", "05", "06", "07",
    "08", "09", "10", "11", "12",
    "13", "14", "15", "16", "17",
    "18", "19", "20", "21", "22",
    "23", "24", "25", "26", "27",
    "28", "29", "30", "31", "32",
]

# IMPORTANT:
# These expanded purpose values are synthetic categories.
# Do not assume every value is an officially supported NPCI code.

# Transaction initiation modes
INITIATION_MODES = [
    "00",  # Default / application-defined
    "01",
    "02",
    "11",

    # Additional synthetic modes for model testing
    "03",
    "04",
    "05",
    "06",
    "07",
    "08",
    "09",
    "10",
    "12",
    "13",
    "14",
    "15",
    "16",
    "17",
    "18",
    "19",
    "20",
]

# Transaction outcome and error codes
RESULT_ERR_CODES = [
    # Existing values
    ("SUCCESS", "01"),
    ("FAILURE", "U30"),
    ("FAILURE", "FR"),

    # Additional illustrative failure scenarios
    ("FAILURE", "U01"),  # Generic simulated failure
    ("FAILURE", "U03"),  # Simulated user/account issue
    ("FAILURE", "U16"),  # Simulated risk rejection
    ("FAILURE", "U17"),  # Simulated PSP issue
    ("FAILURE", "U18"),  # Simulated technical issue
    ("FAILURE", "U29"),  # Simulated timeout
    ("FAILURE", "U31"),  # Simulated bank unavailable
    ("FAILURE", "U69"),  # Simulated timeout / pending case
    ("FAILURE", "U85"),  # Simulated limit-related case
    ("FAILURE", "U90"),  # Simulated connectivity issue
    ("FAILURE", "U91"),  # Simulated issuer unavailable
    ("FAILURE", "U95"),  # Simulated duplicate / replay case
    ("FAILURE", "U96"),  # Simulated system error
    ("FAILURE", "U98"),  # Simulated processing error
    ("FAILURE", "U99"),  # Generic synthetic error
]

# Transaction status categories for eFRM simulation
TRANSACTION_STATUSES = [
    "SUCCESS",
    "FAILURE",
    "PENDING",
    "TIMEOUT",
    "REVERSED",
    "DECLINED",
    "DUPLICATE",
    "UNDER_REVIEW",
]

# Synthetic transaction channels
UPI_CHANNELS = [
    "UPI",
    "UPI_QR",
    "UPI_INTENT",
    "UPI_COLLECT",
    "UPI_P2P",
    "UPI_P2M",
    "UPI_AUTOPAY",
    "UPI_LITE",
    "UPI_OFFLINE",
]

# Synthetic device manufacturers
DEVICE_BRANDS = [
    "Samsung",
    "Xiaomi",
    "Redmi",
    "OnePlus",
    "OPPO",
    "Vivo",
    "Realme",
    "Motorola",
    "Google",
    "Apple",
    "Nothing",
    "iQOO",
    "POCO",
    "Tecno",
    "Infinix",
]

# Network types
NETWORK_TYPES = [
    "4G",
    "5G",
    "WiFi",
    "3G",
    "2G",
    "UNKNOWN",
]

# Synthetic transaction entry modes
ENTRY_MODES = [
    "MOBILE_APP",
    "QR_SCAN",
    "UPI_INTENT",
    "COLLECT_REQUEST",
    "DEEPLINK",
    "USSD",
    "NFC",
    "AUTO_PAY",
]

 
 
def rand_hex(n):
    return ''.join(random.choices(string.hexdigits.lower()[:16], k=n))
 
 
def rand_sha256_like():
    return hashlib.sha256(str(random.random()).encode()).hexdigest() + \
        ''.join(random.choices(string.hexdigits.lower()[:16], k=1))
 
 
def rand_alnum(n):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=n))
 
 
def rand_vpa():
    return f"{rand_sha256_like()[:64]}{random.choice(HANDLES)}"
 
 
def rand_ip():
    return ".".join(str(random.randint(1, 254)) for _ in range(4))
 
 
def rand_msg_id(prefix_len=6, total_len=11):
    return rand_alnum(total_len)
 
 
def rand_device_id():
    return "SUG" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=32))
 
 
def now_ts(base_time):
    return base_time.strftime("%Y-%m-%dT%H:%M:%S") + f"+05:30"
 
 
def rand_account_detail():
    prefix, ifsc = random.choice(BANK_IFSCS)
    return [
        {"name": "ROUTING_IFSC", "value": prefix},
        {"name": "IFSC", "value": ifsc},
        {"name": "ACNUM"},  # masked/absent in sample data
        {"name": "ACTYPE", "value": random.choice(["SAVINGS", "CURRENT"])},
    ]
 
 
def rand_device_tags(with_full_info=True):
    tags = []
    if with_full_info:
        tags.append({"name": "ID", "value": rand_device_id()})
        tags.append({"name": "MOBILE", "value": rand_sha256_like()})
        tags.append({"name": "TYPE", "value": "MOB"})
        tags.append({"name": "OS", "value": random.choice(OS_LIST)})
        tags.append({"name": "APP", "value": random.choice(APPS)})
        tags.append({"name": "IP", "value": rand_ip()})
        tags.append({"name": "GEOCODE"})
        tags.append({"name": "LOCATION"})
        tags.append({"name": "CAPABILITY",
                      "value": ''.join(random.choices(string.digits, k=random.randint(20, 35)))})
        reg = datetime.now(IST) - timedelta(days=random.randint(30, 900))
        rot = reg + timedelta(days=random.randint(1, 400))
        tags.append({"name": "REGISTRATION_TS", "value": now_ts(reg)})
        tags.append({"name": "KEY_ROTATION_TS", "value": now_ts(rot)})
    else:
        tags.append({"name": "MOBILE"})
        tags.append({"name": "TYPE", "value": "MOB"})
        tags.append({"name": "OS", "value": random.choice(OS_LIST)})
    return tags
 
 
def rand_party(amount, is_payer=True):
    party = {
        "type": "PERSON",
        "addr": rand_vpa(),
        "seqNum": "1",
        "code": "0000",
        "Info": {
            "Identity": {"type": "ACCOUNT"},
            "Rating": {"verifiedAddress": "TRUE"},
        },
        "Device": {"Tag": rand_device_tags(with_full_info=is_payer)},
        "Ac": [{"addrType": "ACCOUNT", "Detail": rand_account_detail()}],
        "Amount": {"value": amount, "curr": "INR"},
    }
    if is_payer:
        party["creds"] = {"creds": [{"type": "PIN", "subType": "MPIN"}]}
    return party
 
 
def build_txn_context():
    """Shared identifiers/timestamps reused across the related log lines of one transaction."""
    base_time = datetime.now(IST) - timedelta(minutes=random.randint(0, 60 * 24 * 30))
    amount = f"{random.choice([50, 100, 200, 499, 1000, 2500, 15000]):.2f}"
    return {
        "orgTxnId": "AXL" + rand_hex(32),
        "npciTxnId": "UPI" + rand_alnum(1) + rand_alnum(29) + "npci",
        "note": "P" + ''.join(random.choices(string.digits, k=24)),
        "custRef": ''.join(random.choices(string.digits, k=12)),
        "refId": "T" + datetime.now(IST).strftime("%y%m%d") + rand_alnum(16),
        "ts": now_ts(base_time),
        "amount": amount,
        "req_msg_id_1": rand_msg_id(total_len=11),
        "req_msg_id_2": rand_msg_id(total_len=11),
        "frm_msg_id": "UPI" + rand_alnum(40),
        "base_time": base_time,
    }
 
 
def make_req_pay_message(ctx, response_flag, txn_id, org_txn_id=None, org_resp_code=None):
    txn = {
        "id": txn_id,
        "note": ctx["note"],
        "custRef": ctx["custRef"],
        "refId": ctx["refId"],
        "refUrl": "http://www.npci.org.in/",
        "ts": ctx["ts"],
        "refCategory": "00",
        "purpose": random.choice(PURPOSE_CODES),
        "initiationMode": random.choice(INITIATION_MODES),
        "subType": "PAY",
        "type": "NON-PREAPPROVED",
    }
    if org_txn_id:
        txn["orgTxnId"] = org_txn_id
    if org_resp_code:
        txn["orgRespCode"] = org_resp_code
 
    return {
        "Head": {
            "ver": "2.0",
            "ts": ctx["ts"],
            "orgId": "NPCI",
            "msgId": ctx["req_msg_id_1"] if response_flag == "Y" else ctx["req_msg_id_2"],
            "apiName": "reqPay",
            "mode": "ONLINE",
            "response": response_flag,
        },
        "Txn": txn,
        "Payer": rand_party(ctx["amount"], is_payer=True),
        "Payees": {"Payee": [rand_party(ctx["amount"], is_payer=False)]},
    }
 
 
def make_risk_score_ack(ctx):
    return {
        "api": "ReqRiskScore",
        "reqMsgId": ctx["req_msg_id_1"],
        "err": "",
        "ts": ctx["ts"],
    }
 
 
def rand_score(provider, seq_or_type, extra=None, use_seq=True):
    d = {"provider": provider}
    if use_seq:
        d.update({
            "seqNum": "1",
            "value": f"{random.randint(0, 999):05d}",
            "desc": "",
            "type": seq_or_type,
        })
    else:
        d.update({
            "type": seq_or_type,
            "role": extra,
            "level": None,
            "value": 0,
        })
    return d
 
 
def make_frm_response(ctx, org_txn_id):
    scores = [
        rand_score("NPCI", "DEBITOR"),
        rand_score("NPCI", "CREDITOR"),
    ]
    for t in ["DPS", "ENGAGEMENT", "CONSISTENCY", "SEGMENTATION", "QUALITY"]:
        scores.append(rand_score("NPCI", t, extra="PAYER", use_seq=False))
    for t in ["DPS", "ENGAGEMENT", "CONSISTENCY", "SEGMENTATION", "QUALITY"]:
        scores.append(rand_score("NPCI", t, extra="PAYEE", use_seq=False))
    for role in ["PAYER", "PAYEE", "SendToBoth"]:
        scores.append(rand_score("NPCI", "AML", extra=role, use_seq=False))
 
    result, err_code = random.choice(RESULT_ERR_CODES)
 
    return {
        "Head": {
            "ver": "2.0",
            "ts": ctx["ts"],
            "orgId": "FRM",
            "msgId": ctx["frm_msg_id"],
            "apiName": "reqPay",
            "mode": "ONLINE",
            "response": "Y",
        },
        "Txn": {
            "id": org_txn_id,
            "note": ctx["note"],
            "custRef": ctx["custRef"],
            "refId": ctx["refId"],
            "refUrl": "http://www.npci.org.in/",
            "ts": ctx["ts"],
            "refCategory": "00",
            "purpose": "00",
            "initiationMode": "00",
            "subType": "PAY",
            "type": "NON-PREAPPROVED",
            "RiskScores": {"Score": scores},
        },
        "Resp": {
            "reqMsgId": ctx["req_msg_id_1"],
            "result": result,
            "errCode": err_code,
        },
    }
 
 
def make_frm_advice_ack(ctx):
    return {
        "api": "ReqFrmAdvice",
        "reqMsgId": ctx["req_msg_id_2"],
        "err": "",
        "ts": ctx["ts"],
    }
 
 
def format_log_line(dt, seq_num, tag, payload):
    """Reproduces the pipe-less, colon-separated timestamp header seen in the sample."""
    ts_str = dt.strftime("%d/%b/%Y: %H: %M: %S: ") + f"{seq_num:06d}"
    return f"{ts_str}000000{tag}{json.dumps(payload, indent=2)}"
 
 
def generate_transaction_log():
    """Builds the 5 related log lines for one synthetic UPI transaction."""
    ctx = build_txn_context()
    lines = []
 
    req1 = make_req_pay_message(ctx, response_flag="Y", txn_id=ctx["orgTxnId"])
    lines.append(format_log_line(ctx["base_time"], random.randint(400, 430),
                                  "EFRMrequestmessage: ", req1))
 
    ack1 = make_risk_score_ack(ctx)
    lines.append(format_log_line(ctx["base_time"], random.randint(420, 430),
                                  "EFRMAcknowledgementforReqRiskScore: ", ack1))
 
    frm_resp = make_frm_response(ctx, org_txn_id=ctx["orgTxnId"])
    lines.append(format_log_line(ctx["base_time"], random.randint(560, 570),
                                  "FRMresponsemessage: ", frm_resp))
 
    org_resp_code = "FR" if frm_resp["Resp"]["result"] == "FAILURE" else "00"
    req2 = make_req_pay_message(ctx, response_flag="N", txn_id=ctx["npciTxnId"],
                                 org_txn_id=ctx["orgTxnId"], org_resp_code=org_resp_code)
    lines.append(format_log_line(ctx["base_time"], random.randint(570, 580),
                                  "EFRMrequestmessage: ", req2))
 
    ack2 = make_frm_advice_ack(ctx)
    lines.append(format_log_line(ctx["base_time"], random.randint(575, 580),
                                  "EFRMAcknowledgementforReqFrmAdvice: ", ack2))
 
    return lines
 
 
def generate_dataset(num_transactions=5, out_path=None):
    """Original plain-text log format (5 log lines per transaction)."""
    all_lines = []
    for _ in range(num_transactions):
        all_lines.extend(generate_transaction_log())
    output = "\n".join(all_lines)
    if out_path:
        with open(out_path, "w") as f:
            f.write(output)
    return output
 
 
def build_transaction_json():
    """Builds one synthetic transaction as a JSON-friendly dict (no log-line text wrapper)."""
    ctx = build_txn_context()
 
    req1 = make_req_pay_message(ctx, response_flag="Y", txn_id=ctx["orgTxnId"])
    ack1 = make_risk_score_ack(ctx)
    frm_resp = make_frm_response(ctx, org_txn_id=ctx["orgTxnId"])
 
    org_resp_code = "FR" if frm_resp["Resp"]["result"] == "FAILURE" else "00"
    req2 = make_req_pay_message(ctx, response_flag="N", txn_id=ctx["npciTxnId"],
                                 org_txn_id=ctx["orgTxnId"], org_resp_code=org_resp_code)
    ack2 = make_frm_advice_ack(ctx)
 
    return {
        "reqPay_request": req1,
        "ack_ReqRiskScore": ack1,
        "FRM_response": frm_resp,
        "reqPay_request_leg2": req2,
        "ack_ReqFrmAdvice": ack2,
    }
 
 
def generate_json_dataset(num_transactions=5):
    """Returns a list of synthetic transactions as JSON-serializable dicts."""
    return [build_transaction_json() for _ in range(num_transactions)]
 
 
def save_json_dataset(output_dir, filename="sample_1.json", num_transactions=5):
    """
    Generates synthetic transactions and writes them as a JSON file
    inside output_dir (created if it doesn't exist).
 
    Returns the full path to the written file.
    """
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, filename)
 
    dataset = generate_json_dataset(num_transactions=num_transactions)
    with open(full_path, "w") as f:
        json.dump(dataset, f, indent=2)
 
    return full_path
 
 
def main():
    parser = argparse.ArgumentParser(
        description="Generate synthetic NPCI/UPI FRM transaction data as a JSON file."
    )
    parser.add_argument(
        "-o", "--outdir",
        default=".",
        help="Local directory to save the JSON file into (created if missing). Default: current directory.",
    )
    parser.add_argument(
        "--filename",
        default="sample_1.json",
        help="Output JSON filename. Default: sample_1.json",
    )
    parser.add_argument(
        "-n", "--num",
        type=int,
        default=5,
        help="Number of synthetic transactions to generate. Default: 5",
    )
 
    # parse_known_args ignores unrecognized args (e.g. Jupyter/IPython's
    # injected '-f /path/to/kernel.json'), so this works both from the
    # command line and inside a notebook.
    args, _unknown = parser.parse_known_args()
 
    saved_path = save_json_dataset(
        output_dir=args.outdir,
        filename=args.filename,
        num_transactions=args.num,
    )
    print(f"Saved {args.num} synthetic transactions to: {os.path.abspath(saved_path)}")
 
 
if __name__ == "__main__":
    main()

Saved 5 synthetic transactions to: C:\Users\SARAJIT\Desktop\sample_1.json
